In [ ]:
import sys
import warnings 
from pathlib import Path
from collections import defaultdict
sys.path.append(str(Path.cwd().parents[1]))

import pandas as pd

from source.exp_functions import CVTracker
from configs.uci import FMAP, DATASETS, CV_NAME, RES_NAME, PRED_NAME

warnings.filterwarnings("ignore")

def get_res_df(model: str, hess: str, data_list: list[str]):
    dd = defaultdict(list)
    for data in data_list:
        dd['data'].append(data if data != "wine_red" else "red wine")
        res_dir = Path(f'./artifacts/training_artifacts/{model}/{data}/{hess}/ll/{FMAP}/') 
        try:
            tracker = CVTracker(res_dir, RES_NAME, CV_NAME, PRED_NAME)
            res_df, _, _ = tracker.load()
            res_str = f"{res_df['nll_test'].mean():.2f}±{res_df['nll_test'].std():.2f}"
            dd[model.upper()].append(res_str)
        except:
            dd[model.upper()].append('N/A±0.00')
    return pd.DataFrame(dd).rename({'LA_BTN': 'LA-TNKM'}, axis=1)

In [ ]:
uci_gwi = pd.read_csv('./extra_data/uci_gwi_paper_table.csv', index_col=0)
for model, hess in zip(['mf_btn', 'sp_btn', 'la_btn'], ['mf', 'mf', 'last']):
    res_df = get_res_df(model, hess, DATASETS)
    uci_gwi = pd.merge(uci_gwi, res_df, left_on='Dataset', right_on='data')
    uci_gwi = uci_gwi.drop('data', axis=1)
uci_gwi['Dataset'] = uci_gwi['Dataset'].apply(lambda x: x.upper())
uci_gwi = uci_gwi.drop(['FBNN'], axis=1).rename({'GWI DNN-SVGP': 'GWI-DNN'}, axis=1)
print(uci_gwi)

In [ ]:
caption = (
    r"The average test NLL on several UCI regression datasets. "
    + r"We train on random 90\% of the data and predict on 10\%. "
    + r"This is repeated 10 times and we report mean and standard deviation. "
    + r"$N$ is the sample size and $D$ is the data dimensionality. "
    + r"Among the evaluated methods, LA-TNKM (Last) ranks highest on five of the nine datasets and performs comparably on the rest."
)
print(
    (
        uci_gwi
        .to_latex(
            float_format="%.2f", 
            column_format='||l|c|c||c|c|c|c|c|c|c|c||c||',
            caption=caption,
            label="table:uci-comparison",
            multirow=False,
            index=False,
        ).replace('_', '-')
    )
);